# Семинар 13. SQLite, запросы и схема проекта

Все примеры выполняются на локальной базе в памяти. Мы создадим клиентов и заказы, проверим ограничения, напишем `JOIN` и агрегат, специально сорвём транзакцию и сравним план запроса до и после индекса.

## Цели

После семинара вы сможете:

- открыть SQLite-соединение с явным управлением транзакциями;
- создать таблицы с ключами и ограничениями;
- передавать значения через placeholders;
- читать строки по именам столбцов;
- соединять клиентов с заказами;
- считать агрегаты, не теряя строки без пары;
- подтвердить или откатить связанную операцию;
- проверить внешние ключи и план индекса;
- перенести схему в SQLModel без потери понимания SQL.

## Перед началом

Нужен Python 3.14 и пакет `sqlmodel` только для последнего раздела. Основная часть использует стандартный модуль `sqlite3`. Семинар справочный и рассчитан примерно на 90 минут вместе с обсуждением схем проектов; упражнения не являются отдельной обязательной сдачей.

Перезапуск ячейки создания схемы в той же базе может дать конфликт. Для чистого повторения перезапустите kernel и выполните ноутбук сверху вниз.

## Открываем базу и включаем внешние ключи

`PRAGMA foreign_keys` выполняется до перехода к `autocommit=False`: посреди транзакции переключение не действует. После этого `commit()` и `rollback()` явно завершают текущую транзакцию. `sqlite3.Row` сохраняет доступ и по индексу, и по имени.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:", autocommit=True)
connection.execute("PRAGMA foreign_keys = ON")
assert connection.execute("PRAGMA foreign_keys").fetchone()[0] == 1
connection.autocommit = False
connection.row_factory = sqlite3.Row

> **Появилось в Python 3.12.** `Connection.autocommit` — современный интерфейс управления транзакциями `sqlite3`. В Python 3.14 старое поведение всё ещё является значением по умолчанию, поэтому режим задан явно. В старых учебных примерах вместо него часто настраивается `isolation_level`.

## Создаём схему

Одна строка `customers` — один клиент, одна строка `orders` — один заказ. Email уникален, сумма неотрицательна, статус входит в разрешённый набор, а внешний ключ запрещает заказ неизвестного клиента.

In [ ]:
schema = """
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL CHECK (length(trim(name)) > 0)
);
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0),
    status TEXT NOT NULL CHECK (status IN ('new', 'paid', 'cancelled')),
    comment TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(id) ON DELETE CASCADE
);
"""
try:
    connection.executescript(schema)
    connection.commit()
except Exception:
    connection.rollback()
    raise

## Упражнение 1. Ограничение действительно работает

Для каждого варианта ниже выполните параметризованный `INSERT` внутри `try`, ожидая `sqlite3.IntegrityError`: пустое имя клиента, повтор email, отрицательная сумма, неизвестный статус и заказ с `customer_id=999`. После каждой ошибки откатите транзакцию. Затем объясните, какое именно ограничение сработало.

In [ ]:
def expect_integrity_error(sql: str, params: tuple) -> None:
    try:
        connection.execute(sql, params)
        connection.commit()
    except sqlite3.IntegrityError:
        connection.rollback()
    else:
        raise AssertionError("expected IntegrityError")

# Допишите пять проверок.
...

## Добавляем небольшой набор

Данные намеренно помещаются в две таблицы: имя клиента не копируется в каждый заказ. Сумма хранится в копейках. У Веры нет заказов — эта строка понадобится для проверки `LEFT JOIN`.

In [ ]:
connection.executemany(
    "INSERT INTO customers (email, name) VALUES (?, ?)",
    [
        ("anna@example.test", "Анна"),
        ("boris@example.test", "Борис"),
        ("vera@example.test", "Вера"),
    ],
)
connection.executemany(
    "INSERT INTO orders (customer_id, amount_cents, status, comment) VALUES (?, ?, ?, ?)",
    [
        (1, 900, "new", None),
        (1, 2500, "paid", "доставка вечером"),
        (2, 1200, "paid", None),
    ],
)
connection.commit()

## Пользовательская строка остаётся значением

Параметр с кавычкой и фрагментом SQL не меняет структуру команды. Драйвер ищет буквальный email, а не выполняет дописанное условие. Placeholders защищают значения, но не позволяют динамически передать имя столбца.

In [ ]:
suspicious = "x' OR 1=1 --"
rows = connection.execute(
    "SELECT id, email FROM customers WHERE email = ?",
    (suspicious,),
).fetchall()
assert rows == []

## Упражнение 2. Фильтр и порядок

Напишите один запрос, который получает оплаченные заказы не дешевле переданного `min_amount`, возвращает `id`, `amount_cents`, `comment`, сортирует по убыванию суммы и затем по `id`. Ограничьте результат параметром `limit`. Проверьте `min_amount=1000`, `limit=10`.

In [ ]:
min_amount = 1000
limit = 10
rows = connection.execute(
    # SELECT ... WHERE ... ORDER BY ... LIMIT ...
    ...
).fetchall()
assert [(row["id"], row["amount_cents"]) for row in rows] == [(2, 2500), (3, 1200)]

## `NULL`: проверяем отсутствие явно

У первого и третьего заказов нет комментария. `comment = NULL` не найдёт их, потому что сравнение даёт `UNKNOWN`. Используйте `IS NULL`. Для вывода можно применить `COALESCE(comment, '—')`, но это не меняет сохранённое значение.

In [ ]:
rows = connection.execute(
    "SELECT id, COALESCE(comment, '—') AS display_comment FROM orders WHERE comment IS NULL ORDER BY id"
).fetchall()
assert [(row["id"], row["display_comment"]) for row in rows] == [(1, "—"), (3, "—")]

## Упражнение 3. Все клиенты и их заказы

Через `LEFT JOIN` верните email клиента, id заказа, сумму и статус. Вера должна остаться в результате с `NULL` в полях заказа. Затем измените соединение на `INNER JOIN` и объясните, какая строка пропала и почему.

In [ ]:
joined = connection.execute(
    ...
).fetchall()
assert [(row["email"], row["order_id"]) for row in joined] == [
    ("anna@example.test", 1),
    ("anna@example.test", 2),
    ("boris@example.test", 3),
    ("vera@example.test", None),
]

## Упражнение 4. Статистика

Одним запросом верните email, число всех заказов и сумму только оплаченных. Сохраните Веру с нулями. Используйте `COUNT(orders.id)`, условный `SUM`, `COALESCE`, `GROUP BY` и явный порядок по убыванию суммы, затем по email.

In [ ]:
stats = connection.execute(
    ...
).fetchall()
assert [(row["email"], row["orders_count"], row["paid_total"]) for row in stats] == [
    ("anna@example.test", 2, 2500),
    ("boris@example.test", 1, 1200),
    ("vera@example.test", 0, 0),
]

## Транзакция и настоящий rollback

Создадим два счёта. Функция сначала списывает сумму, затем пытается зачислить её адресату. Если адресата нет, она поднимает исключение внутри `with connection:`; контекстный менеджер откатывает уже выполненное списание. Само соединение после блока остаётся открытым.

In [ ]:
connection.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0))")
connection.executemany("INSERT INTO accounts (id, balance) VALUES (?, ?)", [(1, 1000), (2, 500)])
connection.commit()

def transfer(source_id: int, target_id: int, amount: int) -> None:
    with connection:
        changed = connection.execute(
            "UPDATE accounts SET balance = balance - ? WHERE id = ? AND balance >= ?",
            (amount, source_id, amount),
        )
        if changed.rowcount != 1:
            raise ValueError("source missing or insufficient funds")
        changed = connection.execute(
            "UPDATE accounts SET balance = balance + ? WHERE id = ?",
            (amount, target_id),
        )
        if changed.rowcount != 1:
            raise ValueError("target missing")

try:
    transfer(1, 999, 300)
except ValueError:
    pass
assert [row[0] for row in connection.execute("SELECT balance FROM accounts ORDER BY id")] == [1000, 500]

## Индекс и план запроса

Сравним `EXPLAIN QUERY PLAN` для поиска заказов клиента по статусу. На трёх строках измерять скорость бессмысленно, но план покажет доступную стратегию. После создания составного индекса ищем его имя в `detail`. Индекс ускоряет чтение ценой места и обновления при каждой записи.

In [ ]:
query = "SELECT * FROM orders WHERE customer_id = ? AND status = ?"
before = connection.execute("EXPLAIN QUERY PLAN " + query, (1, "paid")).fetchall()
connection.execute("CREATE INDEX idx_orders_customer_status ON orders (customer_id, status)")
connection.commit()
after = connection.execute("EXPLAIN QUERY PLAN " + query, (1, "paid")).fetchall()
print("before:", [row["detail"] for row in before])
print("after:", [row["detail"] for row in after])
assert any("idx_orders_customer_status" in row["detail"] for row in after)

## Та же идея через SQLModel

Engine создаётся один раз, session — на отдельную операцию. `SQLModel.metadata.create_all` подходит для пустой учебной базы, но не изменяет существующую production-схему как миграция. `select(Customer)` строит SQL-запрос, а `session.exec` выполняет его внутри транзакционного разговора с базой.

In [ ]:
from sqlmodel import Field, Session, SQLModel, create_engine, select

class Customer(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    email: str = Field(unique=True, index=True)
    name: str

engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Customer(email="anna@example.test", name="Анна"))
    session.commit()
with Session(engine) as session:
    customer = session.exec(
        select(Customer).where(Customer.email == "anna@example.test")
    ).one()
    assert customer.name == "Анна"

## Разбор схемы группового проекта

Для каждой таблицы ответьте:

- какой один факт представляет строка;
- каков первичный ключ;
- какие значения обязательны и уникальны;
- какие проверки принадлежат `CHECK`;
- какие внешние ключи и действия удаления нужны;
- какие три запроса будут самыми частыми;
- нужна ли операция из нескольких изменений и где её транзакция;
- какой индекс поддерживает конкретный `WHERE`, `JOIN` или `ORDER BY`;
- как схема будет изменяться после появления реальных данных.

Не начинайте с классов ORM. Сначала нарисуйте факты, связи и запросы; затем отобразите готовую схему на модели.

## Самопроверка

1. Зачем `foreign_keys` включать до транзакции?
2. Какие ограничения находятся в нашей схеме?
3. Почему сумма хранится в копейках?
4. Как placeholder отделяет значение от SQL?
5. Почему `= NULL` не работает?
6. Как сохранить клиента без заказов?
7. Почему `COUNT(*)` после `LEFT JOIN` здесь неверен?
8. Что откатилось при неудачном переводе?
9. Закрывает ли `with connection` соединение?
10. Что изменилось в плане после индекса?
11. Чем engine отличается от session?
12. Почему `create_all` не является миграцией?

## Итоги

- SQLite позволяет воспроизвести ключи, ограничения, запросы и транзакции без отдельного сервера.
- Внешние ключи включаются для каждого соединения.
- Python-значения передаются placeholders, а строки читаются через `row_factory`.
- `LEFT JOIN` сохраняет левую строку, `COUNT(column)` не считает `NULL`.
- Контекст соединения подтверждает или откатывает транзакцию, но не закрывает его.
- `EXPLAIN QUERY PLAN` показывает, получил ли запрос пользу от индекса.
- SQLModel отображает те же таблицы и транзакции на классы; SQL остаётся исполняемой основой.

Домашняя работа объединяет схему, параметризованную запись, `JOIN`, агрегат и атомарную историю смены статуса.